# 04 — Comprehensive Evaluation

**What this notebook does and why it exists**

This is where we find out if all of our work paid off. We load three models — the untuned base model, our SFT+GRPO fine-tuned model, and optionally a larger reference model — and run them on the sacred held-out test set that was never touched during training.

We compute three automatic metrics (BLEU, chrF++, and COMET) and also run an LLM-as-judge A/B evaluation on 50 randomly sampled test pairs. The combination gives us both a comparable number for leaderboard purposes and a qualitative signal about which model actually reads better.

**Why use the sacred test set only now?** Because if we had looked at test set performance during training — to tune hyperparameters, for example — the test set would no longer be a valid estimate of real-world performance. It would have leaked information into our training decisions. The test set is an unbiased estimate of quality precisely because we have never made any decision based on it.

---
## Research Notes: Evaluation Metrics

We use three metrics, chosen for different reasons:

### COMET (Unbabel/wmt22-comet-da) — Primary metric
COMET is a neural metric trained on human quality judgements from WMT shared tasks. It consistently ranks first or second in human correlation across all WMT evaluation campaigns since 2020. Unlike BLEU, it is trained to understand semantic similarity, not surface-level overlap.

**Interpreting COMET scores (wmt22-comet-da scale):**
- Scores are roughly in the range [0.7, 0.9] for decent systems
- **A difference of 1 COMET point (0.01 on the 0–1 scale)** is statistically significant on a test set of 500 segments — it represents a genuinely detectable quality difference
- **A difference of 5 COMET points (0.05)** represents a clear, human-perceivable quality gap — the kind of difference you would notice reading the outputs side by side
- Our fine-tuned model should aim for at least a 2-point improvement over the untuned base model

### chrF++ — Secondary metric  
Character n-gram F-score with word bigrams. Correlates well with human judgement and handles German morphology better than BLEU (because German has rich inflection that changes word endings). Fast to compute — useful during training for quick feedback.

### BLEU — Reported for comparability
The historic standard. We include it because almost every MT paper reports BLEU, making our results comparable to published work. However, for German→English specifically, BLEU is less reliable than chrF++ because it does not handle morphological variants well.

**Why not BLEURT?** BLEURT is a strong metric but requires a separate model download (~3 GB) and is slower. For a comparison of three systems, the cost is justified; for training-time evaluation, it is too slow. We use COMET as the neural metric of record.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install -q \
    transformers==4.45.0 \
    peft==0.13.2 \
    sacrebleu==2.4.3 \
    unbabel-comet==2.2.4 \
    torch==2.4.0 \
    openai==1.54.0 \
    pandas==2.2.2 \
    tabulate==0.9.0

print('Dependencies installed.')

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json, random, time
import pandas as pd

BASE_DIR    = '/content/drive/MyDrive/podcast_translation'
DATA_DIR    = os.path.join(BASE_DIR, 'data')
SFT_ADAPTER = os.path.join(BASE_DIR, 'sft_adapter')
GRPO_DIR    = os.path.join(BASE_DIR, 'grpo_adapter', 'final')

OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
random.seed(42)

BASE_MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
PROMPT_TEMPLATE = (
    'Translate the following German podcast transcript to natural English.\n\n'
    'German: {german}\n\n'
    'Translation:'
)
print('Configuration loaded.')

In [ ]:
# ── Load the sacred test set ──────────────────────────────────────────────────
# THIS IS THE FIRST TIME WE LOOK AT THE TEST SET IN ANY NOTEBOOK.
with open(os.path.join(DATA_DIR, 'test.jsonl'), encoding='utf-8') as f:
    test_data = [json.loads(line) for line in f]

print(f'Test set size: {len(test_data)} pairs')
print(f'\nFirst example:')
print(f'  DE: {test_data[0]["de"]}')
print(f'  EN: {test_data[0]["en"]}')

In [ ]:
# ── Model inference utility ───────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

def load_model(base_name: str, sft_adapter_path: str = None, grpo_adapter_path: str = None):
    """Load a model, optionally applying SFT then GRPO adapters in order."""
    tok = AutoTokenizer.from_pretrained(base_name, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(
        base_name, torch_dtype=torch.bfloat16, trust_remote_code=True
    )
    # Apply SFT adapter first (GRPO was trained on top of merged SFT weights)
    if sft_adapter_path and os.path.isdir(sft_adapter_path):
        mdl = PeftModel.from_pretrained(mdl, sft_adapter_path)
        mdl = mdl.merge_and_unload()
    if grpo_adapter_path and os.path.isdir(grpo_adapter_path):
        mdl = PeftModel.from_pretrained(mdl, grpo_adapter_path)
        mdl = mdl.merge_and_unload()
    mdl.eval()
    return mdl, tok

@torch.no_grad()
def translate_batch(model, tokeniser, german_sentences: list, max_new_tokens=150) -> list:
    """Translate a list of German sentences. Returns list of English strings."""
    translations = []
    for de in german_sentences:
        prompt = PROMPT_TEMPLATE.format(german=de) + ' '
        inputs = tokeniser(
            prompt, return_tensors='pt', truncation=True, max_length=256
        ).to(model.device)
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokeniser.eos_token_id
        )
        gen = tokeniser.decode(
            out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True
        ).strip()
        translations.append(gen)
    return translations

print('Inference utilities defined.')

In [ ]:
# ── Run all models on the test set ────────────────────────────────────────────
# This cell may take 20–60 minutes depending on hardware.
# If running on CPU, consider using only a 50-pair subset first.

german_inputs = [p['de'] for p in test_data]
references    = [p['en'] for p in test_data]

print('=== Loading and running BASE model (untuned) ===')
base_model, base_tok = load_model(BASE_MODEL_NAME)
base_translations = translate_batch(base_model, base_tok, german_inputs)
del base_model   # Free memory
print(f'Base model done. Example: {base_translations[0][:80]}')

print('\n=== Loading and running FINE-TUNED model ===')
ft_model, ft_tok = load_model(BASE_MODEL_NAME, sft_adapter_path=SFT_ADAPTER, grpo_adapter_path=GRPO_DIR)
ft_translations = translate_batch(ft_model, ft_tok, german_inputs)
del ft_model
print(f'Fine-tuned model done. Example: {ft_translations[0][:80]}')

print('\nAll translations complete.')

In [ ]:
# ── Compute automatic metrics ─────────────────────────────────────────────────
import sacrebleu
from comet import download_model, load_from_checkpoint

def compute_bleu_chrf(hypotheses, refs):
    bleu  = sacrebleu.corpus_bleu(hypotheses, [refs])
    chrf  = sacrebleu.corpus_chrf(hypotheses, [refs])
    return bleu.score, chrf.score

# Load COMET model (downloads ~1.5 GB on first run; cached afterwards)
print('Loading COMET model...')
comet_path  = download_model('Unbabel/wmt22-comet-da')
comet_model = load_from_checkpoint(comet_path)

def compute_comet(sources, hypotheses, refs):
    data = [
        {'src': s, 'mt': h, 'ref': r}
        for s, h, r in zip(sources, hypotheses, refs)
    ]
    result = comet_model.predict(data, batch_size=8, gpus=0)
    return result['system_score']

print('Computing metrics for BASE model...')
base_bleu, base_chrf = compute_bleu_chrf(base_translations, references)
base_comet = compute_comet(german_inputs, base_translations, references)

print('Computing metrics for FINE-TUNED model...')
ft_bleu, ft_chrf = compute_bleu_chrf(ft_translations, references)
ft_comet = compute_comet(german_inputs, ft_translations, references)

print('\nMetrics computed.')

In [ ]:
# ── Display results table ─────────────────────────────────────────────────────
from tabulate import tabulate

results = [
    ['Base model (untuned)', f'{base_bleu:.1f}', f'{base_chrf:.1f}', f'{base_comet:.4f}'],
    ['Fine-tuned (SFT+GRPO)', f'{ft_bleu:.1f}', f'{ft_chrf:.1f}', f'{ft_comet:.4f}'],
]

print(tabulate(
    results,
    headers=['Model', 'BLEU ↑', 'chrF++ ↑', 'COMET ↑'],
    tablefmt='github'
))

comet_delta = ft_comet - base_comet
print(f'\nCOMET delta: {comet_delta:+.4f}')
if comet_delta > 0.02:
    print('  → Clear improvement (>2 COMET points). Fine-tuning was effective.')
elif comet_delta > 0.005:
    print('  → Modest improvement. Consider more GRPO steps or a better reward prompt.')
elif comet_delta > -0.005:
    print('  → Roughly equivalent. Fine-tuning maintained quality. GRPO may need tuning.')
else:
    print('  → Regression. Fine-tuned model is worse. Check for reward hacking or overfitting.')

In [ ]:
# ── LLM-as-judge A/B evaluation ───────────────────────────────────────────────
# We sample 50 test pairs and ask gpt-5.2 to compare base vs fine-tuned
# in a blind format (not revealing which is which).
from openai import OpenAI

openrouter_client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=OPENROUTER_API_KEY,
)
JUDGE_MODEL = 'openai/gpt-5.2'   # Verified against openrouter.ai/models

AB_SAMPLE_SIZE = 50
ab_indices = random.sample(range(len(test_data)), AB_SAMPLE_SIZE)

AB_SYSTEM_PROMPT = """You are an expert evaluator of German-to-English translation quality, specialising in podcast and spoken-language content.

You will be given:
  - SOURCE: the original German sentence
  - TRANSLATION A: one English translation
  - TRANSLATION B: another English translation

Choose which translation is better overall, considering:
  1. Accuracy: Does it correctly convey the meaning?
  2. Fluency: Does it sound like natural spoken English?
  3. Register: Is it appropriately conversational (not too formal, not too slangy)?

Respond with ONLY one of: A, B, or TIE
Do not explain your choice."""

wins = {'base': 0, 'finetuned': 0, 'tie': 0}
ab_results = []

for idx in ab_indices:
    de   = test_data[idx]['de']
    ref  = test_data[idx]['en']
    base = base_translations[idx]
    ft   = ft_translations[idx]
    
    # Randomise which is A and which is B to avoid position bias
    if random.random() < 0.5:
        a, b = base, ft
        a_label, b_label = 'base', 'finetuned'
    else:
        a, b = ft, base
        a_label, b_label = 'finetuned', 'base'
    
    user_msg = f'SOURCE: {de}\nTRANSLATION A: {a}\nTRANSLATION B: {b}'
    
    try:
        resp = openrouter_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {'role': 'system', 'content': AB_SYSTEM_PROMPT},
                {'role': 'user', 'content': user_msg},
            ],
            max_tokens=5,
            temperature=0.0,
        )
        verdict = resp.choices[0].message.content.strip().upper()
        if verdict == 'A':
            winner = a_label
        elif verdict == 'B':
            winner = b_label
        else:
            winner = 'tie'
        wins[winner] += 1
        ab_results.append({'de': de, 'base': base, 'finetuned': ft,
                           'ref': ref, 'winner': winner})
    except Exception as e:
        print(f'  Error for index {idx}: {e}')
        time.sleep(2)

total = sum(wins.values())
print(f'\nA/B Evaluation Results ({total} pairs):')
print(f'  Fine-tuned wins: {wins["finetuned"]} ({wins["finetuned"]/total*100:.0f}%)')
print(f'  Base model wins: {wins["base"]} ({wins["base"]/total*100:.0f}%)')
print(f'  Ties:           {wins["tie"]} ({wins["tie"]/total*100:.0f}%)')

In [ ]:
# ── Side-by-side qualitative examples ────────────────────────────────────────
# Show 15 examples from the test set, highlighting interesting cases.
print('=' * 80)
print('SIDE-BY-SIDE EXAMPLES FROM TEST SET')
print('=' * 80)

# Try to show a mix: fine-tuned wins, base wins, ties, and edge cases
sample_indices = (
    [r for r in ab_results if r['winner'] == 'finetuned'][:6] +
    [r for r in ab_results if r['winner'] == 'base'][:4] +
    [r for r in ab_results if r['winner'] == 'tie'][:5]
)[:15]

for i, row in enumerate(sample_indices, 1):
    print(f'\n--- Example {i} (Judge chose: {row["winner"].upper()}) ---')
    print(f'GERMAN:      {row["de"]}')
    print(f'REFERENCE:   {row["ref"]}')
    print(f'BASE MODEL:  {row["base"]}')
    print(f'FINE-TUNED:  {row["finetuned"]}')

In [ ]:
# ── Save results to Drive ─────────────────────────────────────────────────────
eval_output = {
    'automatic_metrics': {
        'base':      {'bleu': base_bleu, 'chrf': base_chrf, 'comet': base_comet},
        'finetuned': {'bleu': ft_bleu,   'chrf': ft_chrf,   'comet': ft_comet},
    },
    'ab_evaluation': wins,
    'comet_delta': float(comet_delta),
}
with open(os.path.join(BASE_DIR, 'evaluation_results.json'), 'w') as f:
    json.dump(eval_output, f, indent=2)

# Save the AB results as CSV for review
ab_df = pd.DataFrame(ab_results)
ab_df.to_csv(os.path.join(BASE_DIR, 'ab_evaluation_details.csv'), index=False)
print('Results saved to Google Drive.')

---
## Summary and Interpretation

### What the numbers typically mean

A well-executed fine-tuning run on a domain-specific task typically yields:
- **COMET improvement:** +0.02 to +0.08 (i.e., 2–8 points on the 0–1 scale)
- **chrF++ improvement:** +1 to +5 points
- **BLEU improvement:** Highly variable; BLEU is sensitive to reference phrasing and can be misleading
- **A/B win rate for fine-tuned:** 55–70% (anything above 50% is a positive signal)

### What to do if results are disappointing

1. **COMET barely moved (< 1 point delta):** The GRPO phase may not have had enough steps, or the KL penalty was too high. Try running more GRPO steps with a lower `KL_COEFF`.
2. **COMET regressed:** Reward hacking during GRPO. Restore the SFT checkpoint and skip GRPO, or re-run GRPO with a stricter KL penalty.
3. **Fine-tuned wins A/B but COMET is flat:** COMET does not fully capture conversational register. Your fine-tuned model may be genuinely better for the podcast use case despite a flat COMET score. Trust the A/B evaluation more in this case.
4. **Base model often wins A/B:** The fine-tuning may have introduced artefacts (truncated outputs, overly formal register). Inspect the examples carefully.

### Further steps if quality is not sufficient

- Collect actual podcast transcripts (even 1,000 segments) and add them to SFT data
- Switch to a 3B or 7B model — the largest model that comfortably runs on your Mac
- Run more GRPO steps with a calibrated reward prompt
- Consider a constitutional approach: add a filter pass that rejects outputs failing basic quality checks before passing them to the LLM judge

**Proceed to: `05_mlx_conversion.ipynb`**